# CANN 是什么

CANN（Compute Architecture for Neural Networks）是华为针对AI场景推出的异构计算架构，对上支持多种AI框架，对下服务AI处理器与编程，发挥承上启下的关键作用，是提升昇腾AI处理器计算效率的关键平台。  

简单来说，CANN就像AI芯片与上层应用之间的“翻译官+调度员”，把开发者写的AI算法代码，转换成芯片能高效执行的指令，同时优化算力分配，最大化AI芯片的性能。

## 1. CANN 在开发链路中的位置

CANN 的核心职责可以概括为三件事：

1. **向下管理硬件**：通过驱动与固件管理昇腾设备，使软件能够访问 NPU 的计算、内存和通信能力。
2. **向上承接框架**：为上层框架和开发工具提供统一接口，使 PyPTO、深度学习框架以及算子库能够在昇腾硬件上运行。
3. **中间完成编译与执行**：将上层计算表达转换为适合目标芯片执行的形式，并负责运行时加载、调度和性能分析。

因此，CANN 不是单一工具，而是一套贯穿开发全流程的软件系统。开发者通常不会在每一步都直接操作 CANN，但 PyPTO 的编译执行、算子调优和运行分析都离不开它。

## 2. 核心组成

CANN 软件栈主要由以下几类组件构成：

### 2.1 驱动与固件（Ascend HDK）

驱动与固件负责连接操作系统和 NPU 硬件，是 CANN 软件栈运行的基础。

- **驱动**：提供设备管理、内存管理、任务提交等底层能力。
- **固件**：负责芯片侧的基础控制和硬件管理。
- **版本要求**：本教程相关内容以 Ascend HDK 25.5.0 及以上版本为基础。

### 2.2 CANN Toolkit 包

Toolkit 是面向开发者的工具集合，覆盖算子开发、编译、调试和调优等环节。

- 提供编译工具链，将上层计算表达转换为目标硬件可执行内容。
- 提供性能分析和调试工具，帮助定位算子瓶颈和执行问题。
- 支持 PyPTO 等开发框架接入 CANN 编译与运行流程。

### 2.3 CANN Ops 包

Ops 包提供常用算子的预置实现，是深度学习模型运行的重要基础。

- 包含常见神经网络算子的高性能实现。
- 支持运行态算子调用和执行。
- 与自定义算子开发形成互补：已有算子可直接复用，特殊场景可通过 PyPTO 等方式扩展。

## 3. 关键能力

CANN 为昇腾开发提供的能力可以从以下几个角度理解：

- **编译能力**：将上层框架或 PyPTO 描述的计算图逐步优化、lowering，并生成适合目标 NPU 执行的代码。
- **运行时能力**：负责模型或算子的加载、任务调度、内存管理和执行控制。
- **算子能力**：通过 Ops 包提供大量预置算子，也支持开发者扩展自定义算子。
- **调试与分析能力**：提供性能分析、执行过程观测和调优辅助工具，帮助开发者发现瓶颈。
- **生态适配能力**：支持主流深度学习框架和昇腾开发工具链，使不同层级的开发方式可以共享同一套硬件基础。

对 PyPTO 学习者来说，最需要关注的是：PyPTO 负责描述计算，CANN 负责把这些计算表达编译、优化并调度到昇腾芯片上执行。

## 4. 架构介绍

### 4.1 什么是异构计算架构
异构计算架构是“使能硬件异构并行计算的软件栈”，最简单的结构就是通用CPU和专用处理器的并行计算组合，其最大的好处是使能多元算力，化解算力瓶颈，从而实现算力最大化。

<img src="./images/heterogeneous_computing_architecture.png" alt="heterogeneous_computing_architecture"  width="350px" >

### 4.2 为什么要用异构计算架构
传统CPU以标量计算为核心，仅能逐元素执行单一运算。而专用硬件可直接完成向量、矩阵级的并行计算，以16×16矩阵乘这一AI场景的核心运算为例，不同计算架构的效率差距极为显著。

<img src="./images/matrix_multiplication_example.png" alt="matrix_multiplication_example"  width="700px" >

在CPU上执行该计算时，每个时钟周期仅能完成一次标量乘加运算，核心计算逻辑如下：
```
for (int i=0; i<16; i++) {
    for (int j=0; j<16; j++) {
        for (int k=0; k<16; k++) {
            // 乘、加操作各占1个时钟周期，单次共需2个cycle
            c[i][j] += a[i][k] * b[k][j];
        }
    }
}
```
**总耗时周期**：Cycle = 16×16×16×2 = 8192。

基于 Vector 矢量硬件单元的专用架构，每个时钟周期可完成一组行与列的矢量乘加运算，核心计算逻辑如下：
```
for (int i=0; i<16; i++) {
    for (int j=0; j<16; j++) {
        // 一行与一列的所有元素同时完成乘加运算
        c[i][j] = a[i][:] *+ b[:][j];
    }
}
```
**总耗时周期**：Cycle = 16×16 = 256。

如果是集成度更高的Cube矩阵乘硬件单元，单个时钟周期即可完成整份矩阵的乘加运算，核心计算逻辑如下：
```
// 两个16×16矩阵一次性完成并行乘加
c[:][:] = a[:][:] * b[:][:];
```
**总耗时周期**：Cycle = 1。

由此可见，面对 AI 场景的密集型计算需求，让专用硬件单元承接核心计算任务，而 CPU 专注于逻辑判断、指令下发等通用任务，可实现 “专人干专事” 的算力最优分配，大幅提升整体计算效率。

### 4.3 CANN架构介绍
CANN正是针对昇腾NPU的**异构计算架构**，它采用分层解耦的设计，上层组件体现CANN的内部能力，下层对接硬件原子能力。通过开放的接口帮助开发者快速调用底层算力，完成计算加速。
- 提供高性能算子及通信算法，帮助开发者进行大模型并行加速，释放芯片澎湃算力
- 提供多种算子开发方式，支持开发者进行高效开发与迁移
- 全面开源，给开发者提供丰富参考实践，让开发者具备自主创新能力

<img src="./images/cann_software_architecture.png" alt="cann_software_architecture"  width="900px" >

## 5. 课后练习

本节练习用于检查 CANN 的定位、核心组成和关键能力。题型包含选择题和填空题，完成后可执行下一单元查看参考答案。

1. （选择题）CANN 在昇腾 AI 开发体系中的定位是什么？  
   A. 只负责展示图片  
   B. 位于上层 AI 框架和底层昇腾硬件之间的软件栈  
   C. 普通文本编辑器  
   D. 数据标注工具

2. （填空题）CANN 的核心组成包括________、________和________等。

3. （选择题）CANN Toolkit 的主要作用是什么？  
   A. 只保存模型权重  
   B. 提供算子开发、编译、调试和调优等工具能力  
   C. 删除运行时环境  
   D. 替代所有硬件驱动

4. （填空题）对 PyPTO 学习者来说，PyPTO 负责描述计算，CANN 负责把计算表达进行________并完成________。

5. （选择题）CANN Ops 包和自定义算子开发的关系是什么？  
   A. 预置算子可直接复用，特殊场景可通过 PyPTO 等方式扩展  
   B. Ops 包只能用于 CPU  
   C. 有 Ops 包就不能开发自定义算子  
   D. Ops 包只负责 Notebook 显示

**执行以下代码获取答案。**


In [ ]:
!cat ./answer/01.03_answer.txt


## 6. 本节小结

CANN 是昇腾 AI 开发的软件底座，向下连接芯片，向上支撑框架和开发工具。它通过驱动固件、Toolkit、Ops 包、编译工具链和运行时环境，把开发者描述的计算变成可以在昇腾芯片上执行的任务。